In [1]:
import pandas as pd
import numpy as np
import re
import json
from pathlib import Path


In [2]:
df = pd.read_csv("../data/processed/smartprix_phones_20261014_042319.csv")

In [3]:
# Shape of Dataset
print(f"Rows: {df.shape[0]}")
print(f"Column: {df.shape[1]}")

Rows: 1021
Column: 13


In [4]:
# The About the Column
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1021 entries, 0 to 1020
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   model         1021 non-null   str  
 1   url           1021 non-null   str  
 2   price         1021 non-null   str  
 3   spec_score    1021 non-null   int64
 4   sim           1010 non-null   str  
 5   processor     955 non-null    str  
 6   ram           1015 non-null   str  
 7   battery       1021 non-null   str  
 8   display       1019 non-null   str  
 9   camera        1018 non-null   str  
 10  card          746 non-null    str  
 11  os            874 non-null    str  
 12  unclassified  377 non-null    str  
dtypes: int64(1), str(12)
memory usage: 103.8 KB


In [5]:
# Null values per Column
df.isnull().sum().sort_values(ascending=False)

unclassified    644
card            275
os              147
processor        66
sim              11
ram               6
camera            3
display           2
model             0
spec_score        0
price             0
url               0
battery           0
dtype: int64

In [6]:
# find the duplicated Models
df.duplicated(subset="model").sum()


np.int64(1)

In [7]:
df[df.duplicated(subset="model", keep=False)][["model", "url", "price", "ram"]]

,model,url,price,ram
0,OPPO F35 Pro,https://smpx.to/jifOik,"₹45,999",NaN
35,OPPO F35 Pro,https://www.smartprix.com/mobiles/oppo-f35-pro...,"₹45,999",NaN


In [8]:
# Convert the Price from text to Integer
df["price_inr"] = df["price"].str.replace(r"[^\d]", "", regex=True).astype("int64")
print(df["price_inr"].describe())

count      1021.000000
mean      38145.505387
std       44544.980185
min         603.000000
25%       14999.000000
50%       26990.000000
75%       43999.000000
max      449900.000000
Name: price_inr, dtype: float64


In [9]:
df.head()

,model,url,price,spec_score,sim,processor,ram,battery,display,camera,card,os,unclassified,price_inr
0,OPPO F35 Pro,https://smpx.to/jifOik,"₹45,999",80,NaN,"Dimensity 7360 Max, Octa Core, 2.5 GHz Processor",NaN,6-Year Durable 8000mAh Battery with 80W SUPERV...,NaN,50MP Ultrawide Natural Selfie Camera,NaN,NaN,IRED Blaster | 1.5K AMOLED Display | 4K Video ...,45999
1,Samsung Galaxy S25 Ultra,https://www.smartprix.com/mobiles/samsung-gala...,"₹94,999",92,"Dual Sim, 5G, VoLTE, Vo5G, NFC, eSIM, Wi-Fi","Snapdragon 8 Elite for Galaxy, Octa Core, 4.47...","12 GB RAM, 256 GB inbuilt",5000 mAh Battery with 45W Fast Charging,"6.9 inches, 1440 x 3120 px, 120 Hz Display wit...",200 MP Quad Rear & 12 MP Front Camera,Memory Card Not Supported,Android v15,NaN,94999
2,OPPO K14 Plus,https://www.smartprix.com/mobiles/oppo-k14-plu...,"₹25,999",75,"Dual Sim, 5G, VoLTE, Wi-Fi, IR Blaster","Dimensity 7360 Max, Octa Core, 2.5 GHz Processor","6 GB RAM, 128 GB inbuilt",8000 mAh Battery with 45W Fast Charging,"6.78 inches, 1272 x 2772 px, 144 Hz Display wi...",50 MP + 2 MP Dual Rear & 16 MP Front Camera,Memory Card Not Supported,Android v16,NaN,25999
3,Motorola Edge 70 Fusion,https://www.smartprix.com/mobiles/motorola-edg...,"₹29,290",70,"Dual Sim, 5G, VoLTE, Vo5G, NFC, Wi-Fi","Snapdragon 7s Gen4, Octa Core, 2.7 GHz Processor","8 GB RAM, 128 GB inbuilt",7000 mAh Battery with 68W Fast Charging,"6.8 inches, 1272 x 2772 px, 144 Hz Display wit...",50 MP + 13 MP + Dedicated 3-in-1 light sensor ...,NaN,Android v16,No FM Radio,29290
4,Vivo V80 5G,https://www.smartprix.com/mobiles/vivo-v80-5g-...,"₹62,999",84,"Dual Sim, 5G, VoLTE, NFC, Wi-Fi, IR Blaster","Snapdragon 7 Gen4, Octa Core, 2.8 GHz Processor","8 GB RAM, 128 GB inbuilt",7200 mAh Battery with 90W Fast Charging,"6.59 inches, 1260 x 2750 px, 144 Hz Display wi...",50 MP + 50 MP + 8 MP Triple Rear & 50 MP Front...,Memory Card Not Supported,Android v17,NaN,62999


In [10]:
# Extracts RAM and storage, converts both into GB


units = {"MB": 1/1024, "GB": 1, "TB": 1024}

# Extract RAM
ram = df["ram"].str.extract(
    r"(\d+(?:\.\d+)?)\s*(MB|GB|TB)\s*RAM",
    flags=re.IGNORECASE
)

df["ram_gb"] = (
    pd.to_numeric(ram[0], errors="coerce")
    * ram[1].str.upper().map(units)
)

# Extract storage separately
storage = df["ram"].str.extract(
    r"(\d+(?:\.\d+)?)\s*(MB|GB|TB)\s*inbuilt",
    flags=re.IGNORECASE
)

df["storage_gb"] = (
    pd.to_numeric(storage[0], errors="coerce")
    * storage[1].str.upper().map(units)
)

# Check missing values
print("RAM missing:", df["ram_gb"].isna().sum())
print("Storage missing:", df["storage_gb"].isna().sum())

# Check RAM distribution
print(df["ram_gb"].value_counts().sort_index().to_string())

RAM missing: 8
Storage missing: 8
ram_gb
0.003906      32
0.007812       6
0.015625       4
0.023438       1
0.031250      48
0.035156       1
0.046875       1
0.048828       1
0.054688       1
0.062500       8
0.500000       9
2.000000       5
3.000000      18
4.000000     182
6.000000     158
8.000000     334
10.000000      1
12.000000    172
16.000000     31


In [11]:
# Check the results
print(df[["ram", "ram_gb", "storage_gb"]].head(10))

                         ram  ram_gb  storage_gb
0                        NaN     NaN         NaN
1  12 GB RAM, 256 GB inbuilt    12.0       256.0
2   6 GB RAM, 128 GB inbuilt     6.0       128.0
3   8 GB RAM, 128 GB inbuilt     8.0       128.0
4   8 GB RAM, 128 GB inbuilt     8.0       128.0
5   6 GB RAM, 128 GB inbuilt     6.0       128.0
6   8 GB RAM, 128 GB inbuilt     8.0       128.0
7   8 GB RAM, 256 GB inbuilt     8.0       256.0
8   6 GB RAM, 128 GB inbuilt     6.0       128.0
9   8 GB RAM, 256 GB inbuilt     8.0       256.0


In [12]:
# 1. Check missing values
print("RAM missing:", df["ram_gb"].isna().sum())
print("Storage missing:", df["storage_gb"].isna().sum())
print("OS missing:", df["os"].isna().sum())

RAM missing: 8
Storage missing: 8
OS missing: 147


In [13]:
# 2. Check whether RAM below 1 GB separates the price groups
low_ram = df["ram_gb"] < 1
high_ram = df["ram_gb"] >= 1

print("\nMaximum price below 1 GB RAM:")
print(df.loc[low_ram, "price_inr"].max())

print("\nMinimum price at least 1 GB RAM:")
print(df.loc[high_ram, "price_inr"].min())


Maximum price below 1 GB RAM:
4899

Minimum price at least 1 GB RAM:
6400


In [14]:
# 3. Check OS for phones with RAM below 1 GB
print("\nOS distribution for phones below 1 GB RAM:")
print(df.loc[low_ram, "os"].value_counts(dropna=False))


OS distribution for phones below 1 GB RAM:
os
NaN             111
KaiOS v2.5.3      1
Name: count, dtype: int64


In [15]:
# 4. Create the feature phone flag
df["is_feature_phone"] = (
    df["ram_gb"].lt(1)
    .where(df["ram_gb"].notna(), pd.NA)
    .astype("boolean")
)

In [16]:
# 5. Compare prediction with actual results
print("\nFeature phone classification among phones with missing OS:")
print(
    df.loc[df["os"].isna(), "is_feature_phone"]
    .value_counts(dropna=False)
)

print("\nOverall classification:")
print(df["is_feature_phone"].value_counts(dropna=False))


Feature phone classification among phones with missing OS:
is_feature_phone
True     111
False     31
<NA>       5
Name: count, dtype: Int64

Overall classification:
is_feature_phone
False    901
True     112
<NA>       8
Name: count, dtype: Int64


# Result 

- 111 feature phones: expected, since they don’t run Android or iOS.
- 31 smartphones with no OS: They’re mostly foldables (Galaxy Z Flip series, Motorola Razr, Vivo X Fold, Apple iPhone Duo) plus some budget brands (Peace, Ringme).
- 5 unknown-RAM rows: the 8 NA rows are the 2 duplicate OPPO F35 Pro cards, the 2 iPhones (Apple doesn’t list RAM), and 4 others. Three of those 4 are obviously feature phones that don’t list RAM (HMD 101 4G, itel Power 450, Lava Hero Shakti). The last is Peace i17 Air, which has an OS but no RAM.

### Resolve the 8 unknown feature-phone flags, investigate missing OS values, and document your decisions ###


In [17]:
# Display the 8 rows

print(df.loc[df["is_feature_phone"].isna(),["model", "price_inr", "ram", "os", "spec_score"]].to_string(index=False))

                       model  price_inr            ram            os  spec_score
                OPPO F35 Pro      45999            NaN           NaN          80
                OPPO F35 Pro      45999            NaN           NaN          80
        Apple iPhone 16 Plus      79900 128 GB inbuilt       iOS v18          70
Apple iPhone 16 Plus (256GB)      89900 256 GB inbuilt       iOS v18          70
                  HMD 101 4G       1799            NaN           NaN          10
               Peace i17 Air       9499            NaN Android v14.0          27
              itel Power 450       1799            NaN           NaN          12
            Lava Hero Shakti        956            NaN           NaN           8


In [18]:

# Apply classifications only to rows that are still unknown

unknown = df["is_feature_phone"].isna()

feature = [
    "HMD 101 4G",
    "itel Power 450",
    "Lava Hero Shakti"
]

smart = [
    "OPPO F35 Pro",
    "Apple iPhone 16 Plus",
    "Apple iPhone 16 Plus (256GB)",
    "Peace i17 Air"
]

df.loc[
    unknown & df["model"].isin(feature),
    "is_feature_phone"
] = True

df.loc[
    unknown & df["model"].isin(smart),
    "is_feature_phone"
] = False

# Verify the result
print(df["is_feature_phone"].value_counts(dropna=False))

print("\nRemaining unknown rows:")
print(
    df.loc[
        df["is_feature_phone"].isna(),
        ["model", "ram", "os"]
    ].to_string(index=False)
)


is_feature_phone
False    906
True     115
Name: count, dtype: Int64

Remaining unknown rows:
Empty DataFrame
Columns: [model, ram, os]
Index: []


In [19]:

# Remove the confirmed duplicate OPPO F35 Pro record

oppo = df[df["model"].eq("OPPO F35 Pro")]

print("OPPO F35 Pro records found:", len(oppo))

if len(oppo) == 2:
    df = df.drop(index=oppo.index[-1]).copy()
    print("Removed one duplicate.")
elif len(oppo) == 1:
    print("No duplicate to remove.")
else:
    print("Check these records manually before proceeding.")

print("\nFinal row count:", len(df))
print(df["is_feature_phone"].value_counts(dropna=False))


OPPO F35 Pro records found: 2
Removed one duplicate.

Final row count: 1020
is_feature_phone
False    905
True     115
Name: count, dtype: Int64


In [20]:
# Identify foldable phones

df["is_foldable"] = df["unclassified"].str.contains(
    "Foldable Display",
    case=False,
    na=False
)

# Compare median prices
print("Foldable counts:")
print(df["is_foldable"].value_counts())

print("\nMedian price by foldable status:")
print(df.groupby("is_foldable")["price_inr"].median())


Foldable counts:
is_foldable
False    982
True      38
Name: count, dtype: int64

Median price by foldable status:
is_foldable
False     25099.5
True     132499.0
Name: price_inr, dtype: float64


In [21]:
# Compare smartphone prices by operating system

os_analysis = (
    df.groupby("os", dropna=False)
    .agg(
        phone_count=("model", "count"),
        median_price=("price_inr", "median"),
        average_price=("price_inr", "mean")
    )
    .sort_values("median_price", ascending=False)
)

print(os_analysis.to_string())

                            phone_count  median_price  average_price
os                                                                  
iOS v27                               8      222400.0  234900.000000
iOS v26                              12      143990.0  148135.666667
Android v17                          22      125499.0  127010.409091
iOS v26.3                             2       78449.5   78449.500000
iOS v11                               3       77060.0   79653.333333
iOS v18                               6       74900.0   84066.666667
iOS v17                               4       69900.0  101924.750000
iOS v15                               1       45690.0   45690.000000
Android v16                         318       32174.0   39451.836478
Android v9.0 (Pie)                   11       31999.0   26362.636364
Android v13                          60       29990.0   34439.616667
Android v14                          90       29249.0   32061.466667
Android v15                       

In [22]:
# Investigate the unclassified specifications

phones = r"Z Flip 7 5G|Peace Nova 17 Pro 4G|Lava Blaze Duo 3 5G"

print(
    df.loc[
        df["model"].str.contains(phones, case=False, na=False),
        ["model", "os", "unclassified"]
    ].to_string(index=False)
)

                                      model  os                   unclassified
                 Samsung Galaxy Z Flip 7 5G NaN Foldable Display, Dual Display
      Peace Nova 17 Pro 4G (6GB RAM + 64GB) NaN                    No FM Radio
                       Peace Nova 17 Pro 4G NaN                    No FM Radio
                        Lava Blaze Duo 3 5G NaN                   Dual Display
Samsung Galaxy Z Flip 7 5G (12GB RAM+512GB) NaN Foldable Display, Dual Display


## Battery And Charging

In [23]:
# Extract battery capacity in mAh

df["battery_mah"] = pd.to_numeric(df["battery"].str.extract(r"(\d+)\s*mAh", flags=re.IGNORECASE)[0],errors="coerce")

# Extract fast-charging wattage

df["fast_charge_w"] = pd.to_numeric(df["battery"].str.extract(r"(\d+(?:\.\d+)?)\s*W\b", flags=re.IGNORECASE )[0],errors="coerce")

# Identify whether fast charging is mentioned
df["fast_charging_supported"] = (df["battery"].str.contains("fast charging", case=False, na=False))

# Check missing values
print("Battery capacity missing:", df["battery_mah"].isna().sum())
print("Fast-charging wattage missing:", df["fast_charge_w"].isna().sum())

Battery capacity missing: 0
Fast-charging wattage missing: 213


In [24]:
# Inspect the original battery text where charging wattage is missing
missing = df["fast_charge_w"].isna()

print(df.loc[missing, "battery"].value_counts(dropna=False).to_string())

battery
1000 mAh Battery                       36
5000 mAh Battery                       15
2000 mAh Battery                       12
2500 mAh Battery                       10
800 mAh Battery                        10
1450 mAh Battery                        9
6000 mAh Battery with Fast Charging     8
3000 mAh Battery                        8
1200 mAh Battery                        7
4000 mAh Battery                        6
5000 mAh Battery with Fast Charging     6
1750 mAh Battery                        5
5391 mAh Battery with Fast Charging     4
4832 mAh Battery with Fast Charging     4
4056 mAh Battery with Fast Charging     4
5800 mAh Battery with Fast Charging     4
4005 mAh Battery with Fast Charging     4
3988 mAh Battery with Fast Charging     3
3149 mAh Battery with Fast Charging     3
1000 mAh Battery with Fast Charging     3
3692 mAh Battery with Fast Charging     2
3349 mAh Battery with Fast Charging     2
4850 mAh Battery with Fast Charging     2
4674 mAh Battery with Fast

In [25]:
print("\nBattery capacity summary:")
print(df["battery_mah"].describe())

print("\nCharging wattage distribution:")
print(df["fast_charge_w"].value_counts(dropna=False).sort_index().to_string())


Battery capacity summary:
count     1020.000000
mean      5286.832353
std       1801.974808
min        600.000000
25%       4830.000000
50%       5100.000000
75%       6500.000000
max      10001.000000
Name: battery_mah, dtype: float64

Charging wattage distribution:
fast_charge_w
2.75        2
10.00      23
15.00      63
18.00      55
20.00       7
23.00       2
25.00      99
27.00       2
30.00      27
33.00      62
37.00       1
44.00      42
45.00     149
50.00      11
60.00      12
65.00       7
66.00       6
67.00      27
68.00      11
80.00     103
90.00      44
100.00     29
120.00     21
125.00      1
150.00      1
NaN       213


In [26]:
# Categorize charging information
mentions_fast = df["battery"].str.contains(
    "fast charging", case=False, na=False
)

has_wattage = df["fast_charge_w"].notna()

df["charging_status"] = "Fast charging, wattage unknown"

df.loc[~mentions_fast, "charging_status"] = (
    "Fast charging not mentioned"
)

df.loc[has_wattage, "charging_status"] = (
    "Fast charging, wattage given"
)

# Create phone-type labels
df["phone_type"] = df["is_feature_phone"].map({
    True: "Feature phones",
    False: "Smartphones"
})

# Compare the groups
print(pd.crosstab(
    df["charging_status"],
    df["phone_type"],
    margins=True
).to_string())

phone_type                      Feature phones  Smartphones   All
charging_status                                                  
Fast charging not mentioned                106           34   140
Fast charging, wattage given                 3          804   807
Fast charging, wattage unknown               6           67    73
All                                        115          905  1020


In [27]:
# Find the smallest smartphone battery
smartphones = df["is_feature_phone"].eq(False)

min_battery = df.loc[smartphones, "battery_mah"].min()

print("Minimum smartphone battery:", min_battery)

print(df.loc[
    smartphones & df["battery_mah"].eq(min_battery),
    ["model", "price_inr", "battery", "os", "url"]
].to_string(index=False))

Minimum smartphone battery: 1821
                 model  price_inr                             battery      os                                                                 url
Apple iPhone 8 (256GB)      77000 1821 mAh Battery with Fast Charging iOS v11 https://www.smartprix.com/mobiles/apple-iphone-8-256gb-p1101j4v0fi0


## ***Os*** ##

In [28]:


os_parts = df["os"].str.extract(
    r"^\s*(Android|iOS|KaiOS)\D*(\d+)?",
    flags=re.IGNORECASE
)

df["os_family"] = os_parts[0].str.lower().map({
    "android": "Android",
    "ios": "iOS",
    "kaios": "KaiOS"
})

df["os_major"] = pd.to_numeric(
    os_parts[1], errors="coerce"
)

print("Missing OS family:", df["os_family"].isna().sum())
print("Missing OS version:", df["os_major"].isna().sum())

# Check the OS value that previously failed
print(
    df.loc[
        df["os"].eq("Android"),
        ["model", "os", "os_family", "os_major"]
    ].to_string(index=False)
)


Missing OS family: 146
Missing OS version: 147
          model      os os_family  os_major
Ringme Bold P70 Android   Android       NaN


In [29]:

# Calculate legacy status
df["is_legacy_os"] = pd.Series(
    pd.NA, index=df.index, dtype="boolean"
)

smartphones = df["is_feature_phone"].eq(False)

android = (
    smartphones
    & df["os_family"].eq("Android")
    & df["os_major"].notna()
)

ios = (
    smartphones
    & df["os_family"].eq("iOS")
    & df["os_major"].notna()
)

df.loc[android, "is_legacy_os"] = (
    df.loc[android, "os_major"] <= 11
)

df.loc[ios, "is_legacy_os"] = (
    df.loc[ios, "os_major"] <= 14
)

# Include smartphones unless they are explicitly classified as legacy.
# Unknown OS/version values remain in scope.
df["in_analysis_scope"] = (
    smartphones
    & ~df["is_legacy_os"].fillna(False)
)

print("Total rows:", len(df))
print("Smartphones:", int(smartphones.sum()))
print(
    "Legacy smartphones:",
    int((smartphones & df["is_legacy_os"].eq(True)).sum())
)
print("In analysis scope:", int(df["in_analysis_scope"].sum()))
print(
    "Unknown legacy status among smartphones:",
    int(df.loc[smartphones, "is_legacy_os"].isna().sum())
)


Total rows: 1020
Smartphones: 905
Legacy smartphones: 67
In analysis scope: 838
Unknown legacy status among smartphones: 33


## ***Display*** ##

In [30]:

# Combine display-related source text
display_text = (
    df["display"].fillna("").astype(str)
    + " "
    + df["unclassified"].fillna("").astype(str)
)

# Screen size in inches
df["screen_in"] = pd.to_numeric(
    display_text.str.extract(
        r'(\d+(?:\.\d+)?)\s*(?:inches|inch|in\b|["″])',
        flags=re.IGNORECASE
    )[0],
    errors="coerce"
)

# Extract resolution dimensions
resolution = display_text.str.extract(
    r"(\d{3,4})\s*[x×]\s*(\d{3,4})",
    flags=re.IGNORECASE
)

res1 = pd.to_numeric(resolution[0], errors="coerce")
res2 = pd.to_numeric(resolution[1], errors="coerce")

# Store shorter and longer dimensions, regardless of order
df["res_short"] = pd.concat(
    [res1, res2], axis=1
).min(axis=1)

df["res_long"] = pd.concat(
    [res1, res2], axis=1
).max(axis=1)

# Refresh rate in Hz, including decimal values
df["refresh_hz"] = pd.to_numeric(
    display_text.str.extract(
        r"(\d+(?:\.\d+)?)\s*Hz\b",
        flags=re.IGNORECASE
    )[0],
    errors="coerce"
)

# Check missing values
display_columns = [
    "screen_in", "res_short", "res_long", "refresh_hz"
]

print("Display column missing-value counts:")
print(df[display_columns].isna().sum())

# Inspect examples where refresh rate is missing
print("\nExamples with missing refresh rate:")
print(
    df.loc[
        df["refresh_hz"].isna(),
        ["model", "display", "unclassified"]
    ].head(15).to_string(index=False)
)


Display column missing-value counts:
screen_in       1
res_short      15
res_long       15
refresh_hz    214
dtype: int64

Examples with missing refresh rate:
                  model                                                        display                                                                                     unclassified
           OPPO F35 Pro                                                            NaN IRED Blaster | 1.5K AMOLED Display | 4K Video on both Rear and Front | 3600 nits Peak Brightness
     Samsung Galaxy F05       6.74 inches, 720 x 1600 px Display with Water Drop Notch                                                                                              NaN
        Apple iPhone 16 6.1 inches, 1179 x 2556 px Display with Dynamic Island Display                                                                                              NaN
Apple iPhone 15 (256GB)         6.1 inches, 1179 x 2556 px Display with Dynamic Island                   

## ***Camera*** ##

In [31]:

# Prepare camera text
camera_text = df["camera"].fillna("").astype(str)

camera_parts = camera_text.str.split(
    "&", n=1, expand=True
)

left = camera_parts[0].fillna("")

if camera_parts.shape[1] > 1:
    right = camera_parts[1].fillna("")
else:
    right = pd.Series("", index=df.index, dtype="string")

has_separator = camera_text.str.contains(
    "&", regex=False, na=False
)

# For text without &, only classify the side when explicitly labelled.
left_has_rear = left.str.contains(
    r"\brear\b", case=False, na=False
)

left_is_front = left.str.contains(
    r"\b(?:front|selfie)\b", case=False, na=False
)

rear_text = left.where(
    (has_separator | left_has_rear) & ~left_is_front,
    ""
)

front_labelled = camera_text.str.contains(
    r"\b(?:front|selfie)\b", case=False, na=False
)

front_text = right.where(
    has_separator,
    camera_text.where(front_labelled, "")
)

# Extract megapixel values
mp_pattern = r"(\d+(?:\.\d+)?)\s*MP\b"

rear_numbers = rear_text.str.findall(
    mp_pattern, flags=re.IGNORECASE
)

front_number = front_text.str.extract(
    mp_pattern, flags=re.IGNORECASE
)[0]

# Create numeric columns
df["rear_main_mp"] = pd.to_numeric(
    rear_numbers.str[0], errors="coerce"
)

df["rear_mp_value_count"] = (
    rear_numbers.str.len()
    .where(rear_text.str.strip().ne(""))
    .astype("Int64")
)

# "No Rear Camera" means a genuine zero
no_rear = camera_text.str.contains(
    r"\bno rear camera\b",
    case=False,
    na=False
)

# A camera is mentioned, but its count is unknown
zero_count = df["rear_mp_value_count"].eq(0).fillna(False)

df.loc[zero_count & ~no_rear, "rear_mp_value_count"] = pd.NA
df.loc[no_rear, "rear_mp_value_count"] = 0

df["rear_mp_value_count"] = df["rear_mp_value_count"].astype("Int64")

print(df["rear_mp_value_count"].value_counts(dropna=False).sort_index())

df["front_mp"] = pd.to_numeric(
    front_number, errors="coerce"
)



rear_mp_value_count
0        36
1       289
2       359
3       317
<NA>     19
Name: count, dtype: Int64


In [32]:
# Check missing values
camera_columns = [
    "rear_main_mp", "rear_mp_value_count", "front_mp"
]

print("Camera column missing-value counts:")
print(df[camera_columns].isna().sum())

# Inspect examples where values were not extracted
for column in camera_columns:
    print(f"\nExamples missing {column}:")
    print(
        df.loc[
            df[column].isna() & df["camera"].notna(),
            ["model", "camera"]
        ].head(10).to_string(index=False)
    )


Camera column missing-value counts:
rear_main_mp            55
rear_mp_value_count     19
front_mp               127
dtype: int64

Examples missing rear_main_mp:
                  model                               camera
           OPPO F35 Pro 50MP Ultrawide Natural Selfie Camera
       Jio Bharat B3 4G                     Main Rear Camera
       Nokia 300 Charge                     Main Rear Camera
Motorola Moto A300 2026                     Main Rear Camera
       Nokia 123 Shield                     QVGA Rear Camera
    Nokia 220 4G (2024)                       No Rear Camera
             HMD 101 4G                       No Rear Camera
  Lava Hero Shakti 2026                       No Rear Camera
   Nokia 130 Music 2023                       No Rear Camera
     Motorola Moto A100                       No Rear Camera

Examples missing rear_mp_value_count:
                  model                               camera
           OPPO F35 Pro 50MP Ultrawide Natural Selfie Camera
      

## ***PROCESSOR*** ##

In [33]:

# Processor extraction with fallback

processor_text = df["processor"].astype("string")

missing_processor = (
    processor_text.isna()
    | processor_text.str.strip().eq("")
)

chip_pattern = (
    r"\b((?:Tensor\s*G\d+[A-Za-z]?|"
    r"Unisoc\s+[A-Za-z0-9-]+|"
    r"Helio\s+[A-Za-z0-9-]+|"
    r"Snapdragon\s+[A-Za-z0-9-]+(?:\s+[A-Za-z0-9-]+){0,3}|"
    r"Dimensity\s+\d+[A-Za-z0-9-]*|"
    r"Exynos\s+\d+[A-Za-z0-9-]*|"
    r"Kirin\s+\d+[A-Za-z0-9-]*|"
    r"Apple\s+A\d+[A-Za-z0-9-]*|"
    r"Bionic\s+A?\d+[A-Za-z0-9-]*))\b"
)

fallback = df["unclassified"].astype("string").str.extract(
    chip_pattern, flags=re.IGNORECASE
)[0]

processor_fallback_used = missing_processor & fallback.notna()

processor_text = processor_text.mask(missing_processor, fallback)

# Processor name
name = processor_text.str.split(",", n=1).str[0].str.strip()

generic = name.str.contains(
    r"\b(?:core|cores|ghz|mhz)\b",
    case=False, na=False
)

df["processor_name"] = name.mask(generic).replace("", pd.NA)

# CPU cores
core_word = processor_text.str.extract(
    r"\b(single|dual|quad|hexa|octa|nine|deca)\s*[- ]?\s*cores?\b",
    flags=re.IGNORECASE
)[0]

core_map = {
    "single": 1, "dual": 2, "quad": 4,
    "hexa": 6, "octa": 8, "nine": 9, "deca": 10
}

df["cpu_cores"] = core_word.str.lower().map(core_map).astype("Int64")

# CPU speed: convert MHz to GHz
ghz = pd.to_numeric(
    processor_text.str.extract(
        r"(\d+(?:\.\d+)?)\s*GHz\b",
        flags=re.IGNORECASE
    )[0], errors="coerce"
)

mhz = pd.to_numeric(
    processor_text.str.extract(
        r"(\d+(?:\.\d+)?)\s*MHz\b",
        flags=re.IGNORECASE
    )[0], errors="coerce"
)

df["cpu_ghz"] = ghz.fillna(mhz / 1000)


# Flag implausibly low processor speeds
suspect_speed = (
    df["cpu_ghz"].notna()
    & (df["cpu_ghz"] < 0.1)
)

print("Suspicious processor speeds:", int(suspect_speed.sum()))

print(
    df.loc[
        suspect_speed,
        ["model", "processor", "cpu_ghz", "in_analysis_scope"]
    ].to_string(index=False)
)

# Treat these source values as unknown, rather than valid speeds
df.loc[suspect_speed, "cpu_ghz"] = np.nan

print("Missing CPU speed after correction:", df["cpu_ghz"].isna().sum())


# Validate
print("Fallbacks used:", int(processor_fallback_used.sum()))
print("Missing processor name:", df["processor_name"].isna().sum())
print("Missing CPU cores:", df["cpu_cores"].isna().sum())
print("Missing CPU speed:", df["cpu_ghz"].isna().sum())


Suspicious processor speeds: 39
                          model                             processor  cpu_ghz  in_analysis_scope
               Jio Bharat B3 4G                       1 MHz Processor    0.001              False
        Motorola Moto A300 2026                     1.5 MHz Processor   0.0015              False
                  Jio Bharat B2                       1 MHz Processor    0.001              False
          Karbonn Jio Bharat A1                       1 MHz Processor    0.001              False
          Lava Hero Shakti 2026                     1.5 MHz Processor   0.0015              False
             Motorola Moto A300                     1.5 MHz Processor   0.0015              False
                 Peace Honor 20 Helio, Single Core, 1.3 MHz Processor   0.0013               True
             Motorola Moto A100               Unisoc, 1 MHz Processor    0.001              False
                  Micromax X702                       1 MHz Processor    0.001        

## ***SIM*** ##

In [34]:
#  SIM FLAGS

sim_text = df["sim"].fillna("").astype("string")
sim_present = sim_text.str.strip().ne("")


def extract_sim_flag(pattern):
    # Unknown if the listing has no SIM text
    result = pd.Series(
        pd.NA,
        index=df.index,
        dtype="boolean"
    )

    # If SIM text exists, True means mentioned; False means not mentioned
    result.loc[sim_present] = (
        sim_text.loc[sim_present]
        .str.contains(pattern, case=False, regex=True, na=False)
        .to_numpy()
    )

    return result


df["is_5g"] = extract_sim_flag(r"\b5G\b")
df["has_nfc"] = extract_sim_flag(r"\bNFC\b")
df["has_esim"] = extract_sim_flag(r"\be-?SIM\b")



## ***Brand*** ##

In [35]:
# BRAND: FIRST WORD OF MODEL


brand_raw = (
    df["model"].astype("string")
    .str.strip()
    .str.extract(r"^(\S+)")[0]
)

brand_key = brand_raw.str.casefold()

# Use the most common original spelling for each normalized brand
brand_lookup = (
    pd.DataFrame({
        "brand_key": brand_key,
        "brand_raw": brand_raw
    })
    .dropna()
    .groupby("brand_key")["brand_raw"]
    .agg(lambda values: values.mode().iloc[0])
)

df["brand"] = brand_key.map(brand_lookup)

# Correct confirmed spelling variants
brand_corrections = int(brand_raw.isin(["Samaung", "Moto"]).sum())

df["brand"] = df["brand"].replace({
    "Samaung": "Samsung",
    "Moto": "Motorola"
})

print("Rows with brand spelling corrections:", brand_corrections)
print("Missing brand:", df["brand"].isna().sum())


Rows with brand spelling corrections: 3
Missing brand: 0


In [36]:
# 4. CHECK MISSING VALUES

new_columns = [
    "processor_name",
    "cpu_cores",
    "cpu_ghz",
    "is_5g",
    "has_nfc",
    "has_esim",
    "brand"
]

print("Missing values in new columns:")
print(df[new_columns].isna().sum().to_string())

print("\nProcessor examples:")
print(
    df[["processor", "processor_name", "cpu_cores", "cpu_ghz"]]
    .head(10)
    .to_string(index=False)
)

print("\nBrand examples:")
print(df[["model", "brand"]].head(10).to_string(index=False))


Missing values in new columns:
processor_name    107
cpu_cores         112
cpu_ghz           169
is_5g              10
has_nfc            10
has_esim           10
brand               0

Processor examples:
                                                   processor                processor_name  cpu_cores  cpu_ghz
            Dimensity 7360 Max, Octa Core, 2.5 GHz Processor            Dimensity 7360 Max          8      2.5
Snapdragon 8 Elite for Galaxy, Octa Core, 4.47 GHz Processor Snapdragon 8 Elite for Galaxy          8     4.47
            Dimensity 7360 Max, Octa Core, 2.5 GHz Processor            Dimensity 7360 Max          8      2.5
            Snapdragon 7s Gen4, Octa Core, 2.7 GHz Processor            Snapdragon 7s Gen4          8      2.7
             Snapdragon 7 Gen4, Octa Core, 2.8 GHz Processor             Snapdragon 7 Gen4          8      2.8
             Snapdragon 6 Gen4, Octa Core, 2.3 GHz Processor             Snapdragon 6 Gen4          8      2.3
             Snap

In [37]:
# 1. Add the scrape date from your actual metadata file
metadata_path = Path(
    "../Data/raw/smartprix_20261014_042319.json"
)

metadata = json.loads(
    metadata_path.read_text(encoding="utf-8")
)

df["scrape_date"] = pd.to_datetime(
    metadata["scraped_at"]
).normalize()

print("Scrape date:", df["scrape_date"].iloc[0].date())


# 2. Classify memory-card support
card_text = df["card"].astype("string").str.strip()

df["has_memory_card"] = pd.Series(
    pd.NA, index=df.index, dtype="boolean"
)

# Explicitly unsupported
not_supported = card_text.str.contains(
    r"\b(?:not supported|unsupported|no memory card)\b",
    case=False,
    na=False
)

# Explicitly supported, including hybrid slots
supported = card_text.str.contains(
    r"\b(?:supported|hybrid|micro\s*sd|memory card)\b",
    case=False,
    na=False
)

df.loc[not_supported, "has_memory_card"] = False

df.loc[
    supported & ~not_supported,
    "has_memory_card"
] = True


# 3. Check classification results
print("\nMemory-card support:")
print(df["has_memory_card"].value_counts(dropna=False))

print("\nOriginal card descriptions:")
print(df["card"].value_counts(dropna=False).head(15).to_string())


Scrape date: 2026-10-14

Memory-card support:
has_memory_card
True     521
<NA>     274
False    225
Name: count, dtype: Int64

Original card descriptions:
card
NaN                                   274
Memory Card Not Supported             225
Memory Card (Hybrid), upto 1 TB        96
Memory Card (Hybrid), upto 2 TB        80
Memory Card (Hybrid)                   76
Memory Card Supported, upto 1 TB       62
Memory Card Supported                  61
Memory Card Supported, upto 32 GB      42
Memory Card Supported, upto 2 TB       34
Memory Card Supported, upto 256 GB     13
Memory Card Supported, upto 512 GB     13
Memory Card Supported, upto 128 GB     11
Memory Card (Hybrid), upto 128 GB       8
Memory Card (Hybrid), upto 1.5 TB       7
Memory Card (Hybrid), upto 512 GB       4


In [38]:
quality_log = []

def log_step(
    step, affected, finding, decision,
    before=None, after=None
):
    quality_log.append({
        "step": step,
        "rows_affected": affected,
        "rows_before": len(df) if before is None else before,
        "rows_after": len(df) if after is None else after,
        "finding": finding,
        "decision": decision
    })


# Historical cleaning steps
log_step(
    "price_inr", 1021,
    "Price values contained currency symbols and separators",
    "Converted prices to numeric INR; kept original text",
    1021, 1021
)

log_step(
    "ram_storage", 8,
    "RAM and storage were unavailable in some listings",
    "Extracted RAM and storage independently; kept unknowns as NaN",
    1021, 1021
)

log_step(
    "is_feature_phone",
    115,
    "112 phones had RAM below 1 GB; the highest price in that group "
    "was ₹4,899, compared with ₹6,400 as the lowest price at 1 GB "
    "RAM or above; 3 additional phones were manually classified",
    "Used the observed price separation to support the RAM threshold "
    "and documented the three model-level decisions",
    1021, 1021
)

log_step(
    "duplicate_removal", 1,
    "Duplicate OPPO F35 Pro listing confirmed",
    "Removed one duplicate",
    1021, 1020
)


# OS and scope
log_step(
    "os_family", 1,
    "146 OS-family values missing; 147 OS versions missing",
    "Recognized family without requiring version; retained unknowns"
)

log_step(
    "is_legacy_os", 67,
    "67 smartphones met the defined legacy-version thresholds",
    "Flagged Android <= 11 and iOS <= 14 without deleting records"
)

log_step(
    "in_analysis_scope", int(df["in_analysis_scope"].sum()),
    "838 smartphones included in the main analysis",
    "Excluded feature phones and explicitly legacy smartphones"
)


# Battery and display
log_step(
    "battery_charging", 213,
    "807 wattages known; 73 fast-charging mentions without wattage; "
    "140 without a fast-charging mention",
    "Kept unknown wattage as NaN; retained charging status"
)

log_step(
    "display_specs", int(df["refresh_hz"].isna().sum()),
    "Missing screen size: 1; missing resolution: 15; missing Hz: 214",
    "Kept unlisted specifications as NaN; did not assume 60 Hz"
)


# Camera
camera_columns = ["rear_main_mp", "rear_mp_value_count", "front_mp"]
camera_missing_rows = int(
    df[camera_columns].isna().any(axis=1).sum()
)

log_step(
    "camera_specs", camera_missing_rows,
    "Missing rear-main MP: 55; rear-camera count: 19; front MP: 127",
    "Kept unavailable values as NaN; explicit No Rear Camera remains zero"
)


# Processor and SIM
processor_missing_rows = int(
    df[["processor_name", "cpu_cores", "cpu_ghz"]]
    .isna().any(axis=1).sum()
)

log_step(
    "processor_specs",
    processor_missing_rows,
    (
        f"Missing processor names: {df['processor_name'].isna().sum()}; "
        f"CPU cores: {df['cpu_cores'].isna().sum()}; "
        f"CPU speed: {df['cpu_ghz'].isna().sum()}"
    ),
    "Used unclassified as a fallback only when processor text was missing; "
    "converted MHz to GHz and mapped Nine Core to 9"
)

log_step(
    "processor_fallback",
    int(processor_fallback_used.sum()),
    f"Recovered {int(processor_fallback_used.sum())} chip names from unclassified",
    "Kept the original processor text as the first source"
)

log_step(
    "memory_card",
    int(df["has_memory_card"].isna().sum()),
    (
        f"Unsupported: {int((df['has_memory_card'] == False).sum())}; "
        f"supported: {int((df['has_memory_card'] == True).sum())}; "
        f"unknown: {int(df['has_memory_card'].isna().sum())}"
    ),
    "Created a nullable Boolean; kept missing descriptions unknown"
)

log_step(
    "scrape_date",
    len(df),
    f"Collection date: {df['scrape_date'].iloc[0].date()}",
    "Added collection date from scraper metadata for future price tracking"
)

sim_columns = ["is_5g", "has_nfc", "has_esim"]
sim_missing_rows = int(
    df[sim_columns].isna().any(axis=1).sum()
)

log_step(
    "sim_features", sim_missing_rows,
    "Each SIM feature flag has 10 missing values",
    "Used NA where SIM text is absent; did not infer unsupported features"
)

log_step(
    "brand", int(df["brand"].isna().sum()),
    "Brand derived from the first word of model names",
    "Standardized casing using the most common spelling per group"
)

log_step(
    "is_foldable", int(df["is_foldable"].sum()),
    "38 listings contained Foldable Display",
    "Created a foldable flag without deleting records"
)

log_step(
    "processor_mhz",
    int(df["processor"].str.contains(
        r"\d+(?:\.\d+)?\s*MHz\b",
        case=False, na=False
    ).sum()),
    "Processor speeds were listed in MHz as well as GHz",
    "Converted MHz to GHz by dividing by 1000"
)

log_step(
    "brand_corrections",
    int(brand_raw.isin(["Samaung", "Moto"]).sum()),
    "Found spelling variants Samaung and Moto",
    "Standardized them to Samsung and Motorola"
)

log_step(
    "cpu_speed_quality",
    39,
    "39 listings produced derived CPU speeds below 0.1 GHz",
    "Set these speeds to NaN pending source verification"
)

quality_log_df = pd.DataFrame(quality_log)

print(quality_log_df.to_string(index=False))

from pathlib import Path

# Notebook reads from ../data/processed, so save there too.
output_dir = Path("../Data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

quality_log_df.to_csv(
    output_dir / "data_quality_log.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nData-quality log saved.")


              step  rows_affected  rows_before  rows_after                                                                                                                                                                                finding                                                                                                             decision
         price_inr           1021         1021        1021                                                                                                                                 Price values contained currency symbols and separators                                                                  Converted prices to numeric INR; kept original text
       ram_storage              8         1021        1021                                                                                                                                      RAM and storage were unavailable in some listings                                         

In [39]:

# Final analytical dataset

final_columns = [
    "model", "url", "brand", "scrape_date",
    "price_inr", "spec_score",
    "processor_name", "cpu_cores", "cpu_ghz",
    "ram_gb", "storage_gb",
    "battery_mah", "fast_charge_w", "charging_status",
    "screen_in", "res_short", "res_long", "refresh_hz",
    "rear_main_mp", "rear_mp_value_count", "front_mp",
    "os_family", "os_major",
    "is_feature_phone", "is_legacy_os", "in_analysis_scope",
    "is_foldable", "is_5g", "has_nfc", "has_esim",
    "has_memory_card"
]

missing_columns = set(final_columns) - set(df.columns)
assert not missing_columns, f"Missing columns: {missing_columns}"

final = df[final_columns].copy()
final = final.reset_index(drop=True)

# Validate the final dataset
assert len(final) == 1020, f"Expected 1020 rows, got {len(final)}"
assert final["model"].is_unique, "Duplicate model names remain"
assert final["price_inr"].notna().all(), "Missing prices remain"
assert final["brand"].notna().all(), "Missing brands remain"
assert final["is_feature_phone"].notna().all()
assert int(final["in_analysis_scope"].sum()) == 838

# Save to the actual project Data/processed folder
from pathlib import Path

output_dir = Path("../Data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

final.to_csv(
    output_dir / "smartphones_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Final shape:", final.shape)
print("Analysis-scope records:", int(final["in_analysis_scope"].sum()))
print("Duplicate models:", final["model"].duplicated().sum())
print("Missing prices:", final["price_inr"].isna().sum())
print("Missing brands:", final["brand"].isna().sum())
print("Saved to:", output_dir / "smartphones_clean.csv")


Final shape: (1020, 31)
Analysis-scope records: 838
Duplicate models: 0
Missing prices: 0
Missing brands: 0
Saved to: ..\Data\processed\smartphones_clean.csv


In [40]:

data_dictionary = [
    ("model", "Smartphone model", "Text", "Required"),
    ("url", "Source product listing", "URL", "Required for traceability"),
    ("brand", "Standardized brand", "Text", "Required"),
    ("scrape_date", "Collection date", "Date", "From scrape metadata"),
    ("price_inr", "Listed price", "INR", "Required"),
    ("spec_score", "Website specification score", "Points", "Source value"),
    ("processor_name", "Processor/chipset name", "Text", "Unknown kept as NaN"),
    ("cpu_cores", "Number of CPU cores", "Cores", "Unknown kept as NaN"),
    ("cpu_ghz","Processor speed","GHz","MHz converted to GHz; suspicious values below 0.1 GHz kept as NaN"),    
    ("ram_gb", "RAM capacity", "GB", "Unknown kept as NaN"),
    ("storage_gb", "Internal storage", "GB", "Unknown kept as NaN"),
    ("battery_mah", "Battery capacity", "mAh", "Extracted from source"),
    ("fast_charge_w", "Charging wattage", "W", "Unknown kept as NaN"),
    ("charging_status", "Charging information status", "Category", "Separates unknown wattage from unmentioned charging"),
    ("screen_in", "Screen size", "Inches", "Unknown kept as NaN"),
    ("res_short", "Short resolution dimension", "Pixels", "Unknown kept as NaN"),
    ("res_long", "Long resolution dimension", "Pixels", "Unknown kept as NaN"),
    ("refresh_hz", "Screen refresh rate", "Hz", "Not assumed to be 60 Hz"),
    ("rear_main_mp", "First identified rear camera resolution", "MP", "Unknown kept as NaN"),
    ("rear_mp_value_count", "Identified rear-camera count", "Cameras", "Zero only when explicitly stated"),
    ("front_mp", "Front camera resolution", "MP", "Unknown kept as NaN"),
    ("os_family", "Operating system family", "Category", "Unknown kept as NaN"),
    ("os_major", "Major OS version", "Version", "Unknown kept as NaN"),
    ("is_feature_phone", "Feature-phone flag", "Boolean", "RAM threshold plus reviewed exceptions"),
    ("is_legacy_os", "Legacy OS flag", "Boolean", "Unknown kept as NA"),
    ("in_analysis_scope", "Included in primary analysis", "Boolean", "Excludes feature phones and identified legacy smartphones"),
    ("is_foldable", "Foldable-display flag", "Boolean", "Based on source label"),
    ("is_5g", "5G mentioned in SIM text", "Boolean", "Missing SIM text remains unknown"),
    ("has_nfc", "NFC mentioned in SIM text", "Boolean", "Missing SIM text remains unknown"),
    ("has_esim", "eSIM mentioned in SIM text", "Boolean", "Missing SIM text remains unknown"),
    ("has_memory_card", "Memory-card support", "Boolean", "Unknown descriptions remain NA"),

]

dictionary_df = pd.DataFrame(
    data_dictionary,
    columns=["column", "meaning", "unit", "missing_handling"]
)

dictionary_df.to_csv(
    output_dir / "data_dictionary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Data dictionary saved:", output_dir / "data_dictionary.csv")


Data dictionary saved: ..\Data\processed\data_dictionary.csv
